In [2]:
# 라이브러리 불러오기
import torch
from transformers import BertTokenizer, BertModel
# 한국어를 위한 버트 토크나이저를 내려받습니다.
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

c:\Users\SSAFY\anaconda3\envs\torch_book\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 문장의 토크나이징
text = '나는 파이토치를 이용한 딥러닝을 학습중이다.'
# 문장의 시작은 [CLS], 문장의 끝은 [SEP]로 형태를 맞춤
marked_text = '[CLS] ' + text + " [SEP]"
# 사전 훈련된 버트 토크나이저를 이용해서 문장을 단어로 쪼갭니다.
tokenized_text = tokenizer.tokenize(marked_text)
print(tokenized_text)

['[CLS]', '나는', '파', '##이', '##토', '##치를', '이', '##용한', '딥', '##러', '##닝', '##을', '학', '##습', '##중', '##이다', '.', '[SEP]']


In [21]:
# 모델을 훈련시킬 텍스트 정의
text = '과수원에 사과가 많았다.'\
        '친구가 나에게 사과했다.'\
        '백설공주는 독이 든 사과를 먹었다.'
# 생성된 문장의 앞에는 [CLS]를 뒤에는 [SEP]를 추가
marked_text = '[CLS] ' + text + ' [SEP]'
# 문장을 토큰으로 분리
tokenized_text = tokenizer.tokenize(marked_text)
# 토크 문자열에 인덱스를 매핑
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
for tup in zip(tokenized_text, indexed_tokens):
    print(f'{tup[0]:<12} {tup[1]:>6,}')


[CLS]           101
과             8,898
##수          15,891
##원에         108,280
사             9,405
##과          11,882
##가          11,287
많             9,249
##았다         27,303
.               119
친             9,781
##구          17,196
##가          11,287
나             8,982
##에게         26,212
사             9,405
##과          11,882
##했다         12,490
.               119
백             9,331
##설          31,928
##공          28,000
##주는         100,633
독             9,088
##이          10,739
든             9,115
사             9,405
##과          11,882
##를          11,513
먹             9,266
##었다         17,706
.               119
[SEP]           102


In [5]:
# 문장 인식 단위 지정
segments_ids = [1] * len(tokenized_text)
print(segments_ids)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [6]:
# 데이터를 텐서로 변환
tokens_tensor = torch.tensor([indexed_tokens])
segments_tensors = torch.tensor([segments_ids])

In [7]:
# 모델 생성
model = BertModel.from_pretrained('bert-base-multilingual-cased', output_hidden_states=True)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [8]:
# 모델 훈련
with torch.no_grad():
    outputs = model(tokens_tensor, segments_tensors)
    # 네트워크 은닉 상태를 가져옵니다.
    hidden_states = outputs[2]

In [9]:
# 모델의 은닉 상태 정보 확인
print('계층 개수:', len(hidden_states), ' (initial embeddings + 12 BERT layers)')
layer_i = 0
print('배치 개수:', len(hidden_states[layer_i]))
batch_i = 0
print('토큰 개수:', len(hidden_states[layer_i][batch_i]))
token_i = 0
print('은닉층의 유닛 개수:', len(hidden_states[layer_i][batch_i][token_i]))

계층 개수: 13  (initial embeddings + 12 BERT layers)
배치 개수: 1
토큰 개수: 33
은닉층의 유닛 개수: 768


In [10]:
# 모델의 은닉 상태 정보 확인
print('은닉 상태의 유형:', type(hidden_states))
print('각 계층에서의 텐서 형태:', hidden_states[0].size())

은닉 상태의 유형: <class 'tuple'>
각 계층에서의 텐서 형태: torch.Size([1, 33, 768])


In [48]:
# 텐서의 형태 변경
# 각 계층의 텐서 결합은 stack을 사용
token_embeddings = torch.stack(hidden_states, dim=0)
# 최종 텐서의 형태를 출력
token_embeddings.size()

torch.Size([13, 1, 33, 768])

In [12]:
# 텐서의 형태 변경
# 배치 차원(1) 제거
token_embeddings = torch.squeeze(token_embeddings, dim=1)
# 배치 차원 제거 후 최종 텐서의 형태를 출력
token_embeddings.size()

torch.Size([13, 33, 768])

In [13]:
# 텐서 차원 변경
token_embeddings = token_embeddings.permute(1, 0, 2)
token_embeddings.size()

torch.Size([33, 13, 768])

In [14]:
# 각 단어에 대한 벡터 형태 확인
# 형태가 [33 x (33 x 768)]인 벡터를 [33 x 25,344]로 변경하여 저장
token_vecs_cat = []
# token_embeddings는 [33 x 12 x 768] 형태의 텐서를 갖습니다.
for token in token_embeddings:
    cat_vec = torch.cat((token[-1], token[-2], token[-3], token[-4]), dim=0)
    token_vecs_cat.append(cat_vec)
print(f'형태는: {len(token_vecs_cat)} x {len(token_vecs_cat[0])}')

형태는: 33 x 3072


In [15]:
# 계층을 결합하여 최종 단어 벡터 생성
# [33 x 768] 형태의 토큰을 벡터로 저장
token_vecs_sum = []
# 'token_embeddings'는 [33 x 12 x 768] 형태의 토큰을 갖습니다.
for token in token_embeddings:
    # 마지막 4개 계층의 벡터를 합산
    sum_vec = torch.sum(token[-4:], dim=0)
    token_vecs_sum.append(sum_vec)
    # 'sum_vec'를 사용하여 토큰을 표현
print(f'형태는: {len(token_vecs_sum)} x {len(token_vecs_sum[0])}')

형태는: 33 x 768


In [16]:
# 문장 벡터
token_vecs = hidden_states[-2][0]
sentence_embedding = torch.mean(token_vecs, dim=0)
print('최종 임베딩 벡터의 형태:', sentence_embedding.size())

최종 임베딩 벡터의 형태: torch.Size([768])


In [17]:
# 토큰과 인덱스 출력
for i, token_str in enumerate(tokenized_text):
    print(i, token_str)

0 [CLS]
1 과
2 ##수
3 ##원에
4 사
5 ##과
6 ##가
7 많
8 ##았다
9 .
10 친
11 ##구
12 ##가
13 나
14 ##에게
15 사
16 ##과
17 ##했다
18 .
19 백
20 ##설
21 ##공
22 ##주는
23 독
24 ##이
25 든
26 사
27 ##과
28 ##를
29 먹
30 ##었다
31 .
32 [SEP]


In [18]:
# 단어 벡터 확인, 가독성을 위해 5개만 가져 옴
print('사과가 많았다', str(token_vecs_sum[6][:5]))
print('나에게 사과했다', str(token_vecs_sum[10][:5]))
print('사과를 먹었다', str(token_vecs_sum[19][:5]))

사과가 많았다 tensor([-0.5844, -4.0836,  0.4906,  0.8915, -1.8054])
나에게 사과했다 tensor([-0.8631, -3.4047, -0.7351,  0.9805, -2.6700])
사과를 먹었다 tensor([ 0.6756, -0.3618,  0.0586,  2.2050, -2.4193])


In [19]:
# 코사인 유사도 계산
from scipy.spatial.distance import cosine

# '사과가 많았다'와 '나에게 사과했다'에서 단어 '사과' 사이의 코사인 유사도를 계산
diff_apple = 1 - cosine(token_vecs_sum[5], token_vecs_sum[27])
# '사과가 많았다'와 '사과를 먹었다'에 있는 '사과' 사이의 코사인 유사도를 계산
same_apple = 1 - cosine(token_vecs_sum[5], token_vecs_sum[16])
print(f'*유사한* 의미에 있는 벡터 유사성: {same_apple:.2f}')
print(f'*다른* 의미에 있는 벡터 유사성: {diff_apple:.2f}')

*유사한* 의미에 있는 벡터 유사성: 0.86
*다른* 의미에 있는 벡터 유사성: 0.91
